# HW11: find the intended word
T1 needs only NumPy and Matplotlib. T2-T3 download `words.zip` unless it is already beside this notebook; no extraction is needed. The helper plots your table but does not compute edit distance.

## T1. Build the edit-distance matrix
Run the display demo first: its interior numbers are invented, not edit distances. Then use the same helper on your own `DIST` for `causality` to `casualty`. The starter sets the empty-prefix boundaries only. You supply the recurrence and the traceback. No download is needed for T1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def show_edit_table(source, target, table, title, path=()):
    """Print and plot supplied values. NaN means unfilled; path cells are optional."""
    d = np.asarray(table, dtype=float)
    if d.shape != (len(source) + 1, len(target) + 1) or np.isinf(d).any():
        raise ValueError("Use one row and column per prefix, including empty prefixes; use NaN for blanks.")
    rows = ["empty"] + [source[:i] for i in range(1, len(source) + 1)]
    cols = ["empty"] + [target[:j] for j in range(1, len(target) + 1)]
    row_width = max(map(len, rows))
    width = max(map(len, cols)) + 1
    print(" " * row_width + "".join(f"{c:>{width}}" for c in cols))
    for word, row in zip(rows, d):
        values = ("." if np.isnan(v) else f"{v:g}" for v in row)
        print(f"{word:>{row_width}}" + "".join(v.rjust(width) for v in values))

    fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
    image = ax.imshow(np.ma.masked_invalid(d), cmap="GnBu", vmin=0)
    ax.set_xticks(range(len(cols)), labels=cols, rotation=40, ha="left")
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_yticks(range(len(rows)), labels=rows)
    ax.set(xlabel=f"Target: {target}", ylabel=f"Source: {source}")
    fig.suptitle(title)
    for (i, j), value in np.ndenumerate(d):
        if np.isnan(value):
            continue
        r, g, b, _ = image.cmap(image.norm(value))
        color = "white" if 0.299*r + 0.587*g + 0.114*b < 0.48 else "black"
        ax.text(j, i, f"{value:g}", ha="center", va="center", color=color)
    for i, j in path:
        ax.add_patch(Rectangle((j - 0.46, i - 0.46), 0.92, 0.92,
                               fill=False, edgecolor="#bf4700", linewidth=2))
    fig.colorbar(image, ax=ax, label="Cell value", shrink=0.75)
    plt.show()

In [ ]:
# Display example only. The interior values below are deliberately made up.
demo_DIST = [[0, 1, 2, 3],
             [1, 8, 2, 7],
             [2, 5, 9, 4]]
show_edit_table("ab", "xyz", demo_DIST, "Display demo: invented values, not edit distances")

### Your turn: causality to casualty
The first row and column count edits to or from an empty prefix. Each interior cell starts as `NaN` (a dot in the printout and blank on the heatmap). Replace `pass` with your recurrence, rerun the cell, then trace one optimal path. The helper displays your numbers; it does not check their correctness.

In [ ]:
source, target = "causality", "casualty"

def boundary_matrix(source, target):
    d = np.full((len(source) + 1, len(target) + 1), np.nan)
    d[0, :] = np.arange(len(target) + 1)
    d[:, 0] = np.arange(len(source) + 1)
    return d

DIST = boundary_matrix(source, target)
for i in range(1, len(source) + 1):
    for j in range(1, len(target) + 1):
        # TODO: assign DIST[i, j] using your unit-cost recurrence.
        pass

path = []  # Add your traceback as (row, column) pairs.
show_edit_table(source, target, DIST, "Your unit-cost matrix", path)
if np.isfinite(DIST).all():
    print("Your calculated distance:", DIST[-1, -1])

### Change one rule
Choose **A:** allow a swap of adjacent letters at cost 1, or **B:** charge 3 for a substitution and do not allow swaps. Keep other costs unchanged. Predict what will happen, recompute your matrix, and compare one changed cell or traceback decision with your unit-cost result.

In [ ]:
DIST_modified = boundary_matrix(source, target)
for i in range(1, len(source) + 1):
    for j in range(1, len(target) + 1):
        # TODO: assign DIST_modified[i, j] using the rule you chose.
        pass

modified_path = []  # Add a traceback for this rule.
show_edit_table(source, target, DIST_modified, "Your modified matrix", modified_path)
if np.isfinite(DIST_modified).all():
    print("Your modified distance:", DIST_modified[-1, -1])

## T2. Compare three models
Use the same query set for each model and pass only the two strings and the model to your distance function. You can use integer cost units internally (e.g. 5 units for cost 1) to make ties exact. The three counters below are mutually exclusive.

In [ ]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile, is_zipfile

PAGE = "https://courses.cs.ut.ee/2026/algorithmics/fall/Main/HW11"

def course_zip(name):
    path = Path(name)
    if not path.is_file():
        url = f"{PAGE}?action=download&upname={name}"
        try:
            with urlopen(url, timeout=45) as response:
                data = response.read()
        except OSError as error:
            raise RuntimeError(f"Download {url} and place {name} beside this notebook.") from error
        if not is_zipfile(BytesIO(data)):
            raise ValueError(f"Expected a ZIP from {url}; download it manually.")
        path.write_bytes(data)
    return ZipFile(path)

with course_zip("words.zip") as archive:
    levels = {level: archive.read(f"words.{level}.txt").decode("utf-8").splitlines()
              for level in range(7)}
originals = levels[0]
assert all(len(words) == 100 and all(words) for words in levels.values())
assert len(set(originals)) == len(originals)
print(f"{len(originals)} dictionary entries; corruption levels 1 to 6.")

In [ ]:
from collections import Counter

query_rows = list(range(0, 100, 10))
test_levels = (1, 3, 6)

def outcome(winners, expected):
    winners = set(winners)
    if not winners:
        raise ValueError("Return all minimum-cost words, not an empty set.")
    if expected not in winners:
        return "miss"
    return "unique correct" if len(winners) == 1 else "tie including original"

def evaluate(search, model, level):
    # search(query, dictionary, model) returns an iterable of tied best words.
    counts = Counter()
    for row in query_rows:
        winners = search(levels[level][row], originals, model)
        counts[outcome(winners, originals[row])] += 1
    return dict(counts)

assert outcome(["abc"], "abc") == "unique correct"
assert outcome(["abc", "cab"], "cab") == "tie including original"
assert outcome(["abc"], "cab") == "miss"
print("Queries per level:", len(query_rows))

In [ ]:
# Implement distance(source, target, model) and search(query, dictionary, model).
# Check T1, empty words, case-only change, a similar-letter substitution and a swap.
# Example after implementation: evaluate(search, "unit", 1)

## T3. Find the five best matches in a longer text
In T2 both ends of each word had to align. Here the **whole query** must align, but the text before or after the match is free. Write `best_match(query, segment)` using your unit-cost DP. Return `(cost, start, end)` for one best **nonempty** substring of that segment, with zero-based, end-exclusive indices. Break ties consistently.

Test first: `best_match("cat", "xxcutyy")` should return `(1, 2, 5)`. Use your T2 function to check the cost of each returned substring. The reporting helper takes care of selecting five separate locations; it never joins text across a previously reported match.

In [ ]:
query = originals[0]
text = "".join(word for level in range(1, 7) for word in levels[level])
print(f"Query: {query} | Search text: {len(text)} characters (no separators)")

# Implement best_match(query, segment) here. Only the text outside the match is free.
# assert best_match("cat", "xxcutyy") == (1, 2, 5)

In [ ]:
def top_five(query, text, best_match):
    """Use your one-match DP on the unreported parts; return original text positions."""
    pending, hits = [], []

    def consider(lo, hi):
        if lo < hi:
            cost, start, end = best_match(query, text[lo:hi])
            if not (0 <= start < end <= hi - lo):
                raise ValueError("Return a nonempty interval relative to the input segment.")
            pending.append((cost, lo + start, lo + end, lo, hi))

    consider(0, len(text))
    while pending:
        hit = min(pending)  # Lowest cost, then leftmost among the segment winners.
        pending.remove(hit)
        cost, start, end, lo, hi = hit
        hits.append((cost, start, end, text[start:end]))
        if len(hits) == 5:
            break
        consider(lo, start)
        consider(end, hi)
    return hits

def print_hits(hits):
    print("rank | edit cost | start | end | matched text")
    for rank, (cost, start, end, content) in enumerate(hits, 1):
        print(f"{rank:>4} | {cost:>9g} | {start:>5} | {end:>3} | {content}")

# Once your one-match DP works:
# hits = top_five(query, text, best_match)
# print_hits(hits)
# Optional: run the same helper with your weighted or swap-aware matcher.